# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Method: Logistic Regression (primary) + Random Forest (comparison)

I'm choosing logistic regression as my primary method because its coefficients directly answer Lane 1's question — which signals associate with decline, and in which direction, with a magnitude I can read and defend. It's also the natural next step after Week 3's leakage-check model, which already used logistic regression to get an honest baseline AUC of 0.557 on five signals.

I'm adding random forest for comparison because Week 2's framing argued ML beats a fixed rule specifically because signals interact in combinations a linear model or if-statement can't capture (e.g. moderate impressions + old content + weak engagement). Random forest can pick up those interactions; comparing it against logistic regression tells me whether nonlinear combinations actually add value here, or whether a simple linear model is already capturing what matters.

I'm not using clustering — Lane 1's question is about association with a specific outcome (decline), not about discovering unlabeled groups, which is Lane 3's job. Gradient boosting is available in the menu, but I'm holding it in reserve unless random forest clearly underperforms and complexity is worth it — the assignment explicitly warns against rewarding complexity alone.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Client-holdout (grouped by client_hash_id)

I'm splitting by client, not by row — holding out ~20% of clients entirely for testing, so no client's pages appear in both train and test. This matters because pages from the same client likely share patterns (writing style, SEO strategy, industry) that a model could partially memorize rather than genuinely learn from signals alone. A random row-level split would let the model "cheat" by recognizing a client's style from other pages of theirs already seen in training — this is exactly the reasoning the reference pipeline uses (GUIDE.md, Section 2: "the split holds out ~20% of clients").

I'm not using a time-aware split here, because my label (is_declining, from Week 3/4) compares the first half vs. second half of the same March window — it's not a genuine past→future prediction task yet (Week 2 flagged this proxy-label weakness explicitly). A time-aware split would be the right choice for a future-looking label, which is a natural next step beyond this week's scope.

In [6]:
import numpy as np

# Get unique clients from the March data
clients_df = con.sql(f"""
    SELECT DISTINCT client_hash_id
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
""").df()

np.random.seed(42)
all_clients = clients_df["client_hash_id"].values
n_holdout = int(len(all_clients) * 0.20)
holdout_clients = set(np.random.choice(all_clients, size=n_holdout, replace=False))

print(f"Total clients: {len(all_clients)}")
print(f"Holdout clients: {len(holdout_clients)}")
print(f"Train clients: {len(all_clients) - len(holdout_clients)}")

Total clients: 47
Holdout clients: 9
Train clients: 38


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW model_data AS
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS total_sessions,
        SUM(scroll_events) AS total_scroll_events,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""")

model_df = con.sql("SELECT * FROM model_data").df()
model_df["is_declining"] = (model_df["imp_second_half"] < model_df["imp_first_half"]).astype(int)

print("Shape:", model_df.shape)
model_df.head()

Shape: (176738, 10)


,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_scroll_events,imp_first_half,imp_second_half,is_declining
0,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,1.0,0.0,4173.0,2350.0,1
1,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,3.0,0.0,3705.0,1925.0,1
2,content_b00348592e2becad,client_73cda7b4e4f265ea,96.0,0.0,0.654533,0.0,0.0,13.0,83.0,0
3,content_71de1df9da7ad732,client_73cda7b4e4f265ea,867.0,2.0,7.687704,1.0,0.0,451.0,416.0,1
4,content_ca2c5289ed4504fd,client_73cda7b4e4f265ea,209.0,0.0,47.087924,0.0,0.0,85.0,124.0,0


In [8]:
train_df = model_df[~model_df["client_hash_id"].isin(holdout_clients)].copy()
test_df = model_df[model_df["client_hash_id"].isin(holdout_clients)].copy()

print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train decline rate:", train_df["is_declining"].mean())
print("Test decline rate:", test_df["is_declining"].mean())

Train rows: 125409 | Test rows: 51329
Train decline rate: 0.3619038506008341
Test decline rate: 0.41302187847026045


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

feature_cols = ["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events"]

train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

X_train, y_train = train_df[feature_cols], train_df["is_declining"]
X_test, y_test = test_df[feature_cols], test_df["is_declining"]

# Logistic Regression (scaled, since raw impressions have a huge range)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)
logreg_pred = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_auc = roc_auc_score(y_test, logreg_pred)

# Random Forest (no scaling needed)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)

print(f"Logistic Regression AUC (client-holdout test): {logreg_auc:.3f}")
print(f"Random Forest AUC (client-holdout test): {rf_auc:.3f}")

Logistic Regression AUC (client-holdout test): 0.538
Random Forest AUC (client-holdout test): 0.576


In [10]:
test_df["ctr"] = 100.0 * test_df["total_clicks"] / test_df["total_impressions"]
test_df["baseline_flag"] = (
    (test_df["total_impressions"] >= 500) &
    (test_df["avg_position"] > 0) & (test_df["avg_position"] <= 20) &
    (test_df["ctr"] < 0.5)
).astype(int)

baseline_auc = roc_auc_score(y_test, test_df["baseline_flag"])
print(f"Week 4 baseline (low_ctr_visible_page flag) AUC on same test set: {baseline_auc:.3f}")

Week 4 baseline (low_ctr_visible_page flag) AUC on same test set: 0.521


Model vs. baseline comparison (same client-holdout test set)

| Method | ROC AUC |
|---|---:|
| Baseline (`low_ctr_visible_page` flag) | 0.521 |
| Logistic Regression | 0.538 |
| Random Forest | **0.576** |

Random Forest beats both the baseline (+0.055) and logistic regression (+0.038), supporting
Week 2's argument that combinations of signals matter more than any single threshold rule.
However, all three numbers sit close to 0.5 (random guessing) — this isn't a strong discovery,
it's confirmation of what Week 3 and Week 4 already suggested: these five signals (impressions,
clicks, position, sessions, scroll events), on their own, carry only weak information about
within-month decline. The improvement from baseline to Random Forest is real but modest, and
I'm reporting it honestly rather than treating a 0.576 AUC as a strong result.

This also validates the client-holdout split's importance: a weaker validation design (e.g.
random row split) might have shown inflated numbers by letting the model partially memorize
client-specific patterns — the modest gap here across all three methods suggests the split is
doing its job of testing genuine generalization, not client memorization.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.